## 라이브러리 호출 및 데이터 로드

In [22]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

In [23]:
# 데이터 로드
df_match = pd.read_csv('./유저단위_게임데이터_상위랭커보존-데이터복구완료.csv')

In [24]:
df_match.info()

<class 'pandas.DataFrame'>
RangeIndex: 396239 entries, 0 to 396238
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gameid               396239 non-null  str    
 1   gameduration         396239 non-null  float64
 2   level                396239 non-null  int64  
 3   lastround            396239 non-null  int64  
 4   ranked               396239 non-null  int64  
 5   ingameduration       396239 non-null  float64
 6   combination          396239 non-null  str    
 7   champion             396239 non-null  str    
 8   user_tier            396239 non-null  str    
 9   season               396239 non-null  str    
 10  user_id              396239 non-null  str    
 11  flag_1               396239 non-null  int64  
 12  flag_2               396239 non-null  int64  
 13  mix_flag             396239 non-null  str    
 14  combination_rebuild  396239 non-null  str    
dtypes: float64(2), int64(5), str

---
## 1. 시너지명 정리
- 시너지명에 'Set3_'가 포함된 경우, 삭제

In [25]:
# combination_rebuild 컬럼의 데이터 타입을 딕셔너리로 변경
df_match['combination_rebuild'] = df_match['combination_rebuild'].apply(ast.literal_eval)

# 데이터형이 변경되었는지 확인
type(df_match['combination_rebuild'].iloc[0]) # dict

dict

In [26]:
# 전체 키 목록 확인하는 함수 정의
# set(): 중복을 허용하지 않는 자료형(집합)
# .update(): set에 여러 값을 한 번에 추가하는 메서드
def get_all_keys(df, col):
    all_keys = set()
    for value in df[col]:
        all_keys.update(value.keys())
    return all_keys

# 함수 실행 (접두사 'Set3_' 제거하기 전)
print(get_all_keys(df_match, 'combination_rebuild'))

{'Set3_Mystic', 'Set3_Celestial', 'Infiltrator', 'Set3_Void', 'Set3_Sorcerer', 'Set3_Brawler', 'Blaster', 'Set3_Blademaster', 'Chrono', 'StarGuardian', 'Sniper', 'Starship', 'Mercenary', 'Cybernetic', 'Valkyrie', 'Rebel', 'DarkStar', 'ManaReaver', 'SpacePirate', 'MechPilot', 'Demolitionist', 'Vanguard', 'Protector'}


In [27]:
# 접두사인 'Set3_'를 삭제하는 함수 정의
def remove_set3_prefix(x):
    result = {}
    for key, value in x.items():
        if 'Set3_' in key:
            new_key = key.replace('Set3_', '')
        else:
            new_key = key
        result[new_key] = value
    return result

In [28]:
# 정의한 함수를 사용해 시너지명 정리
# apply(): 행마다 함수를 적용함
df_match['combination_rebuild'] = df_match['combination_rebuild'].apply(remove_set3_prefix)

In [ ]:
# 접두사 'Set3_' 제거한 결과 확인
print(get_all_keys(df_match, 'combination_rebuild'))

{'Brawler', 'Sorcerer', 'Infiltrator', 'Blaster', 'Chrono', 'Celestial', 'StarGuardian', 'Sniper', 'Starship', 'Mercenary', 'Void', 'Cybernetic', 'Valkyrie', 'Blademaster', 'Rebel', 'DarkStar', 'ManaReaver', 'Mystic', 'SpacePirate', 'MechPilot', 'Demolitionist', 'Vanguard', 'Protector'}


---
## 2. 비활성화된 시너지 삭제
- 게임이 진행되는 동안, 활성화되지 않은 시너지 정보 삭제

In [30]:
# 시너지별 활성화 기준(Threshold) 정의
synergy_thresholds = {
    # 직업 시너지
    'Blademaster': [3, 6, 9],
    'ManaReaver': [2],
    'Sorcerer': [2, 4, 6, 8],
    'Vanguard': [2, 4],
    'Protector': [2, 4, 6],
    'Mystic': [2, 4],
    'Brawler': [2, 4],
    'Mercenary': [1],
    'Starship': [1],
    'Infiltrator': [2, 4, 6],
    'Sniper': [2],
    'Blaster': [2, 4],
    'Demolitionist': [2],
    
    # 계열 시너지
    'Void': [3],
    'MechPilot': [3],
    'Rebel': [3, 6, 9],
    'Valkyrie': [2],
    'StarGuardian': [3, 6],
    'Cybernetic': [3, 6],
    'Chrono': [2, 4, 6],
    'DarkStar': [3, 6, 9],
    'SpacePirate': [2, 4],
    'Celestial': [2, 4, 6]
}

In [31]:
# 활성화 레벨 계산 함수 정의
def get_active_level(count, thresholds):
    active = 0
    for t in thresholds:
        if count >= t:
            active = t
        else:
            break
    return active

# 딕셔너리(dict) 형태로 활성화된 시너지만 반환하는 함수 정의
def parse_active_synergies_to_dict(comb_dict):
    # 데이터가 이미 딕셔너리 형태이므로 ast.literal_eval 불필요. 빈 딕셔너리인지 확인
    if not isinstance(comb_dict, dict) or not comb_dict:
        return {}
        
    active_res = {}
    for syn, count in comb_dict.items():
        if syn in synergy_thresholds:
            active_lvl = get_active_level(count, synergy_thresholds[syn])
            # 활성화 레벨이 0보다 큰 경우(최소 조건 달성)에만 딕셔너리에 추가
            if active_lvl > 0:
                active_res[syn] = active_lvl
                
    # 결과도 깔끔하게 딕셔너리 형태로 반환 (json 변환 불필요)
    return active_res

In [ ]:
# 'active_synergies' 컬럼을 추가하여 활성화된 시너지 딕셔너리만 따로 확인
df_match['active_synergies'] = df_match['combination_rebuild'].apply(parse_active_synergies_to_dict)

In [33]:
# 결과 확인
display(df_match[['combination_rebuild', 'active_synergies']].head())

,combination_rebuild,active_synergies
0,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Brawler': 1, 'Celestial': 1, 'Void': 1, 'Sniper': 1}",{}
1,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Blademaster': 2, 'Brawler': 1, 'Sorcerer': 1, 'Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Cybernetic': 3, 'Vanguard': 2}"
2,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Blademaster': 1, 'Mystic': 1, 'Valkyrie': 1}",{'Mercenary': 1}
3,"{'DarkStar': 1, 'Protector': 2, 'Blademaster': 1, 'Celestial': 5, 'Mystic': 1, 'Sniper': 1, 'StarGuardian': 2, 'Vanguard': 2}","{'Protector': 2, 'Celestial': 4, 'Vanguard': 2}"
4,"{'Blaster': 1, 'Chrono': 5, 'DarkStar': 3, 'Protector': 1, 'Blademaster': 1, 'Brawler': 1, 'Sorcerer': 2, 'Sniper': 2}","{'Chrono': 4, 'DarkStar': 3, 'Sorcerer': 2, 'Sniper': 2}"
